# DSCC 201/401 Final Project - Question1

L. EXTRA CREDIT 2 (10 points): Use the PyTorch library as provided by the Python 3 (Python 3 (PyTorch 1.9.1)) kernel to build a simple artificial neural network for the prediction of the solar radio flux based on the sunspot number and the individual number of X-ray and optical solar flares. Begin with importing your cleaned data file (solar_data_clean.csv) into a Panadas data frame. Partition the data set appropriately and train your neural network. What is the predicted radio flux for a day when 73 sunspots are observed on the sun and 2 solar flares with C-class energy levels and 1 solar flare with an M-class energy level occur? How confident are you in the answer? Explain your reasoning. Please provide a PDF of your Jupyter notebook showing ALL input and output and upload BOTH the PDF and the Jupyter notebook file (ExtraCredit2.ipynb and ExtraCredit2.pdf) showing all input and output.

In [57]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Load the cleaned solar data file
df = pd.read_csv("solar_data_clean.csv")
df.head()

,Date,Radio_Flux_10.7_cm,SESC_Sunspot_Number,Sunspot_Area,New_Regions,GOES15_XRay_Bkgd_Flux,XRay_C,XRay_M,XRay_X,S,Optical_1,Optical_2,Optical_3
0,1997-01-01,72,0,0,0,A0.5,0,0,0,1,0,0,0
1,1997-01-02,72,0,0,0,A0.5,0,0,0,1,0,0,0
2,1997-01-03,73,0,0,0,A0.5,0,0,0,1,0,0,0
3,1997-01-04,74,13,10,1,A0.6,0,0,0,1,0,0,0
4,1997-01-05,74,15,20,0,A0.7,0,0,0,1,0,0,0


In [58]:
# Select feature and target columns
feature_cols = [
    "SESC_Sunspot_Number",
    "XRay_C", 
    "XRay_M", 
    "XRay_X",
    "S",
    "Optical_1", 
    "Optical_2", 
    "Optical_3"
]

target_col = "Radio_Flux_10.7_cm"

X = df[feature_cols].to_numpy()
y = df[target_col].to_numpy()

# Partition the data set into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Convert NumPy arrays to PyTorch tensors
X_train_t = torch.from_numpy(X_train_scaled).float()
y_train_t = torch.from_numpy(y_train).float().view(-1, 1)

X_test_t = torch.from_numpy(X_test_scaled).float()
y_test_t = torch.from_numpy(y_test).float().view(-1, 1)

print("Train tensors:", X_train_t.shape, y_train_t.shape)
print("Test tensors :", X_test_t.shape, y_test_t.shape)

# Create TensorDataset and DataLoader
train_set = TensorDataset(X_train_t, y_train_t)
test_set  = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=32, shuffle=False)

Train tensors: torch.Size([8433, 8]) torch.Size([8433, 1])
Test tensors : torch.Size([2109, 8]) torch.Size([2109, 1])


In [59]:
# Build a neural network model
n_features = X_train_t.shape[1]

class SolarNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

model = SolarNet(n_features)
print(model)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

SolarNet(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [60]:
# Train the neural network model
epochs = 100

for epoch in range(epochs):
    
    # Training phase
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)

        loss.backward()
        optimizer.step()

        train_loss = train_loss + loss.item() * X_batch.size(0)

    train_loss = train_loss / len(train_loader.dataset)

    # Validate phase
    model.eval()
    val_loss = 0
    val_mae = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            y_pred = model(X_batch)

            mse_loss = criterion(y_pred, y_batch)

            abs_error = torch.abs(y_pred - y_batch)
            mae_loss = torch.mean(abs_error)

            val_loss = val_loss + mse_loss.item() * X_batch.size(0)
            val_mae  = val_mae  + mae_loss.item() * X_batch.size(0)

    val_loss = val_loss / len(test_loader.dataset)
    val_mae  = val_mae  / len(test_loader.dataset)
    print("Epoch", epoch + 1, "/", epochs, 
          f"Train MSE: {train_loss:.4f}  "
          f"Val MSE: {val_loss:.4f}  "
          f"Val MAE: {val_mae:.4f}")

Epoch 1 / 100 Train MSE: 12185.0119  Val MSE: 5401.6982  Val MAE: 66.8959
Epoch 2 / 100 Train MSE: 2008.1520  Val MSE: 870.1762  Val MAE: 20.6937
Epoch 3 / 100 Train MSE: 692.6682  Val MSE: 588.7899  Val MAE: 16.4865
Epoch 4 / 100 Train MSE: 520.9130  Val MSE: 462.7639  Val MAE: 14.4552
Epoch 5 / 100 Train MSE: 431.1289  Val MSE: 386.3536  Val MAE: 13.1436
Epoch 6 / 100 Train MSE: 377.4093  Val MSE: 341.7519  Val MAE: 12.3849
Epoch 7 / 100 Train MSE: 344.2419  Val MSE: 313.3672  Val MAE: 11.8798
Epoch 8 / 100 Train MSE: 322.6823  Val MSE: 293.7842  Val MAE: 11.3969
Epoch 9 / 100 Train MSE: 309.4405  Val MSE: 279.9771  Val MAE: 11.1616
Epoch 10 / 100 Train MSE: 298.0881  Val MSE: 270.8099  Val MAE: 10.8698
Epoch 11 / 100 Train MSE: 291.1228  Val MSE: 264.3270  Val MAE: 10.7208
Epoch 12 / 100 Train MSE: 285.7601  Val MSE: 263.7703  Val MAE: 10.7937
Epoch 13 / 100 Train MSE: 282.1133  Val MSE: 256.2530  Val MAE: 10.7123
Epoch 14 / 100 Train MSE: 279.6626  Val MSE: 253.1686  Val MAE: 10.65

In [62]:
# Evaluate the Model Performance on the test set
model.eval()
with torch.no_grad():
    pred_test = model(X_test_t)
    mse_test = torch.mean((pred_test - y_test_t) ** 2).item()
    mae_test = torch.mean(torch.abs(pred_test - y_test_t)).item()

print("PyTorch Test MSE:", mse_test)
print("PyTorch Test MAE:", mae_test)

PyTorch Test MSE: 216.53500366210938
PyTorch Test MAE: 10.211377143859863


In [63]:
 # Use the given features to predict the solar radio flux
x_new = np.array([[73, 2, 1, 0, 0, 0, 0, 0]])
x_new_scaled = scaler.transform(x_new)
x_new_t = torch.from_numpy(x_new_scaled).float()

# Predict using the trained model
model.eval()
with torch.no_grad():
    predicted_radio_flux = model(x_new_t).item()

print("Predicted radio flux:", predicted_radio_flux)

Predicted radio flux: 116.77032470703125


In [64]:
# Compute 95% heuristic prediction interval

margin = 1.96 * mae_test
lower_CI = predicted_radio_flux - margin
upper_CI = predicted_radio_flux + margin
print(f"95% prediction interval: [{lower_CI:.2f}, {upper_CI:.2f}]")

95% prediction interval: [96.76, 136.78]


Using the trained neural network model, **the predicted solar radio flux** for a day when 73
sunspots are observed on the sun and 2 solar flares with C-class energy levels and 1 solar flare with
an M-class energy level occur is **approximately 116.77**. The neural network achieves **a MAE of
approximately 10.21 on the test set**, which provides an empirical estimate of the prediction error
on unseen data. Because neural networks do not provide statistical standard errors, a statistical
prediction interval cannot be constructed as in linear regression model. Instead, the test-set
MAE was used to construct **a heuristic prediction interval**, which is **roughly 96.76 to 136.78**.
I am **moderately confident** in the prediction.